# Capítulo 4 — Projetos de Tabelas inteligentes: Por que ser normal?

**Livro:** começa na página **169** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(4, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Duas tabelas de pescadores · PDF p. 170

#### Exercício: Mesma pergunta, duas tabelas

Encontre os peixes da Bahia nas duas tabelas: em `info_peixes` o estado está misturado em `local` (`LIKE '%, BA'`); em `recordes_peixes` há a coluna `uf`. Qual consulta é mais confiável?

In [ ]:
%%sql
SELECT 'info_peixes' AS tabela, nome_comum, local FROM info_peixes WHERE local LIKE '%, BA'
UNION ALL
SELECT 'recordes_peixes', nome_comum, local || ' / ' || uf FROM recordes_peixes WHERE uf = 'BA';

## Dados atômicos · PDF p. 175

#### Exercício: O preço de não ser atômico

Em `meus_contatos`, a coluna `interesses` guarda vários valores. Encontre quem gosta de `xadrez` e de `cinema` ao mesmo tempo. Repare como o `LIKE` é frágil (e se alguém escrever `Xadrez`?).

In [ ]:
%%sql
SELECT primeiro_nome, interesses FROM meus_contatos
WHERE interesses LIKE '%xadrez%' AND interesses LIKE '%cinema%';

## Regras da CHAVE PRIMÁRIA · PDF p. 182

#### Exercício: Chave primária impede repetidos

Crie `clientes` com `cliente_id INT PRIMARY KEY` e `nome`. Insira o cliente 1 e tente inserir outro cliente 1 (`try/except`).

In [ ]:
%sql CREATE TABLE clientes (cliente_id INT PRIMARY KEY, nome VARCHAR(30) NOT NULL)
%sql INSERT INTO clientes VALUES (1, 'Ana')
try:
    %sql INSERT INTO clientes VALUES (1, 'Outra Ana')
except Exception as erro:
    print("Chave repetida bloqueada:", "duplicate key" in str(erro) or "duplicada" in str(erro))
%sql SELECT * FROM clientes

## Consertando a tabela do Greg · PDF p. 184

#### Exercício: Separando cidade e estado

Sem mudar a tabela, mostre `local` separado em `cidade` e `uf` usando `split_part(local, ', ', 1)` e `split_part(local, ', ', 2)`. (No capítulo 5 você grava isso em colunas.)

In [ ]:
%%sql
SELECT primeiro_nome, split_part(local, ', ', 1) AS cidade, split_part(local, ', ', 2) AS uf
FROM meus_contatos;

## Mostre-me a tabela · PDF p. 186

#### Exercício: SHOW CREATE TABLE do PostgreSQL

O MySQL tem `SHOW CREATE TABLE`. Aqui, junte as colunas (`information_schema.columns`) com as restrições (`information_schema.table_constraints`) de `clientes`. Liste as restrições: tipo e nome.

In [ ]:
%%sql
SELECT constraint_type AS tipo, constraint_name AS nome
FROM information_schema.table_constraints
WHERE table_schema = current_schema() AND table_name = 'clientes'
ORDER BY 1;

## 1, 2, 3... auto incrementando · PDF p. 188

#### Exercício: Chave que se numera sozinha

Crie `contatos_novos` com `contato_id SERIAL PRIMARY KEY` (o `AUTO_INCREMENT` do MySQL), `primeiro_nome` e `email`. Insira 3 pessoas **sem** informar o id e mostre a tabela.

In [ ]:
%%sql
CREATE TABLE contatos_novos (
    contato_id SERIAL PRIMARY KEY,
    primeiro_nome VARCHAR(20) NOT NULL,
    email VARCHAR(50)
);
INSERT INTO contatos_novos (primeiro_nome, email) VALUES
    ('Ana', 'ana@exemplo.com'), ('Bruno', 'bruno@exemplo.com'), ('Carla', NULL);
SELECT * FROM contatos_novos;

## ALTER TABLE e adicionar uma PRIMARY KEY · PDF p. 192

#### Exercício: Chave primária numa tabela que já existe

Acrescente `contato_id SERIAL PRIMARY KEY` à tabela `meus_contatos` (que já tem dados) com `ALTER TABLE ... ADD COLUMN`. Os ids são preenchidos sozinhos? (No PostgreSQL a coluna nova vai para o fim; o `FIRST` do MySQL não existe.)

In [ ]:
%%sql
ALTER TABLE meus_contatos ADD COLUMN contato_id SERIAL PRIMARY KEY;
SELECT contato_id, primeiro_nome, sobrenome FROM meus_contatos ORDER BY contato_id;